# KNN Classifier — Hotel Booking Cancellation Prediction

Author: Siriwardana N.D.V.S

KNN (K-Nearest Neighbors) is a supervised, non-parametric, instance-based lazy learner. Unlike other models, it stores the entire training set and classifies new bookings by majority vote among the K most similar training examples using Euclidean or Manhattan distance.

The objective of this notebook is to predict whether a hotel booking will be cancelled (`is_canceled = 1`) or not (`is_canceled = 0`) using the Hotel Booking Demand dataset, and compare KNN performance against the group's other models (Logistic Regression, Decision Tree, Random Forest).


In [1]:
import os
import sys

# Resolve project root even if notebook is launched from a nested folder.
current_dir = os.path.abspath(os.getcwd())
project_root = None

# Search parent directories for src/config.py as the project anchor.
for _ in range(6):
    config_path = os.path.join(current_dir, "src", "config.py")
    if os.path.exists(config_path):
        project_root = current_dir
        break
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

# Stop early with a clear error if root cannot be discovered.
if project_root is None:
    raise RuntimeError(
        "Could not find src/config.py within 5 parent levels. "
        "Open VS Code from the project root and rerun this cell."
    )

# Use project root as cwd so relative artifact/data paths stay stable.
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Working directory set to: {os.getcwd()}")

Working directory set to: d:\SLIIT\Y4S2\IT4060 - Machine Learning\Assignment\repo\Machine-Learning-Assignment


In [2]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.use("Agg")

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report
from sklearn.inspection import permutation_importance
import sklearn

# --- Shared project modules (consistent workflow with other model notebooks) ---
from src import config
from src.data_loader import load_hotel_bookings, basic_train_ready_checks
from src.preprocessing import build_preprocessor, PreprocessOptions, get_feature_names
from src.train_eval import (
    TrainOptions,
    split_xy,
    make_train_test_split,
    get_estimator,
    build_model_pipeline,
    tune_with_gridsearch,
    predict_with_optional_proba,
    evaluate_on_test,
)
from src.metrics import compute_classification_metrics, format_metrics_for_print
from src.plots import plot_confusion_matrix, plot_roc_curve, plot_pr_curve
from src.io_utils import (
    ensure_artifact_dirs,
    save_json,
    save_text,
    save_dataframe,
    save_model,
    save_run_metadata,
)

# Create artifact directories before any write operations.
DIRS = ensure_artifact_dirs()
print("Artifact directories:")
for name, path in DIRS.items():
    print(f"- {name}: {path}")

# Log environment details for reproducibility in reports.
print(f"Python version: {sys.version}")
print(f"scikit-learn version: {sklearn.__version__}")

Artifact directories:
- base: artifacts
- data: artifacts\data
- preprocessing: artifacts\preprocessing
- models: artifacts\models
- metrics: artifacts\metrics
- plots: artifacts\plots
- reports: artifacts\reports
Python version: 3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]
scikit-learn version: 1.8.0


## 1. Data Loading

The dataset is loaded using the project's `data_loader` utility, preferring the deduplicated processed version if available, with a fallback to the raw dataset path from config.


In [3]:
# --- Data loading strategy ---
# Prefer processed deduplicated data; fallback to default configured path.
processed_path = os.path.join("data", "processed", "hotel_bookings_dedup.csv")

if os.path.exists(processed_path):
    df = load_hotel_bookings(processed_path, drop_duplicates=False, verbose=True)
    print("Loaded deduplicated dataset from processed folder")
else:
    df = load_hotel_bookings(
        config.DEFAULT_DATA_PATH, drop_duplicates=True, verbose=True
    )
    print("Processed file not found, loaded from raw path with deduplication")

# Run basic guards: target presence, shape sanity, train readiness.
basic_train_ready_checks(df, target_col="is_canceled")

# Enforce categorical treatment for these identifier-like columns.
for col in ["agent", "company"]:
    if col in df.columns:
        df[col] = df[col].astype(str)

print(f"Dataset shape: {df.shape}")

# Quick class-balance snapshot for imbalanced-class decisions later (F1, threshold tuning).
class_counts = df["is_canceled"].value_counts(dropna=False).sort_index()
class_perc = (
    df["is_canceled"].value_counts(normalize=True, dropna=False).sort_index() * 100
).round(2)

print("Class distribution (is_canceled):")
for label in class_counts.index:
    print(f"  {label}: {int(class_counts[label])} ({class_perc[label]:.2f}%)")

print("Confirmed: 'agent' and 'company' cast to str")

[data_loader] Loaded shape: (87396, 32)
[data_loader] Columns: 32
Loaded deduplicated dataset from processed folder
Dataset shape: (87396, 32)
Class distribution (is_canceled):
  0: 63371 (72.51%)
  1: 24025 (27.49%)
Confirmed: 'agent' and 'company' cast to str


In [4]:
# --- Train/test split ---
# Separate features/target, then stratify to preserve class ratio in both sets.
X, y = split_xy(df, target_col="is_canceled")

opts = TrainOptions(test_size=0.20, random_state=42)
X_train, X_test, y_train, y_test = make_train_test_split(X, y, options=opts)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape} | y_test shape: {y_test.shape}")

train_counts = y_train.value_counts(dropna=False).sort_index()
train_perc = (
    y_train.value_counts(normalize=True, dropna=False).sort_index() * 100
).round(2)
test_counts = y_test.value_counts(dropna=False).sort_index()
test_perc = (
    y_test.value_counts(normalize=True, dropna=False).sort_index() * 100
).round(2)

print("Class distribution in y_train:")
for label in train_counts.index:
    print(f"  {label}: {int(train_counts[label])} ({train_perc[label]:.2f}%)")

print("Class distribution in y_test:")
for label in test_counts.index:
    print(f"  {label}: {int(test_counts[label])} ({test_perc[label]:.2f}%)")

# Store class counts as plain ints for JSON serialization.
train_dist = {str(k): int(v) for k, v in y_train.value_counts().to_dict().items()}
test_dist = {str(k): int(v) for k, v in y_test.value_counts().to_dict().items()}

# --- Persist split metadata for traceability and model comparison notebook ---
split_metadata = {
    "model": "knn",
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
    "test_ratio": 0.20,
    "random_state": 42,
    "stratified": True,
    "train_class_distribution": train_dist,
    "test_class_distribution": test_dist,
}

split_meta_path = os.path.join("artifacts", "data", "train_test_split_knn.json")
save_json(split_metadata, split_meta_path)
print("Split metadata saved to artifacts/data/train_test_split_knn.json")

X_train shape: (69916, 31) | y_train shape: (69916,)
X_test shape: (17480, 31) | y_test shape: (17480,)
Class distribution in y_train:
  0: 50696 (72.51%)
  1: 19220 (27.49%)
Class distribution in y_test:
  0: 12675 (72.51%)
  1: 4805 (27.49%)
Split metadata saved to artifacts/data/train_test_split_knn.json


## 2. Preprocessing Pipeline & Model Construction

The preprocessing pipeline uses `build_preprocessor()` with `output_sparse=False` because KNN requires dense matrix input and sparse matrices can raise a `TypeError`. The full workflow encapsulates preprocessing and the KNN estimator in a single sklearn `Pipeline` with steps named `preprocess` and `model`. All preprocessing parameters are fitted exclusively on the training set to prevent data leakage.


In [5]:
# --- Preprocessing + model pipeline ---
# KNN needs dense arrays; sparse matrices can fail during distance computations.
preprocess_opts = PreprocessOptions(output_sparse=False)

# Pull configurable column rules (with safe defaults).
drop_cols = getattr(
    config,
    "LEAKAGE_COLS",
    ["reservation_status", "reservation_status_date"],
)
force_categorical_cols = getattr(
    config,
    "FORCE_CATEGORICAL_COLS",
    ["agent", "company"],
)

# Build preprocessing exactly once and embed into sklearn pipeline.
preprocessor = build_preprocessor(
    drop_cols=drop_cols,
    force_categorical_cols=force_categorical_cols,
    options=preprocess_opts,
)

# Get base KNN estimator from shared factory for consistency.
estimator = get_estimator("knn")
pipeline = build_model_pipeline(preprocessor, estimator)

print("Pipeline steps:")
for step_name, step_obj in pipeline.named_steps.items():
    print(f"- {step_name}: {type(step_obj).__name__}")

print("\nBase KNN estimator parameters:")
print(estimator.get_params())

Pipeline steps:
- preprocess: Pipeline
- model: KNeighborsClassifier

Base KNN estimator parameters:
{'algorithm': 'auto', 'leaf_size': 30, 'metric': 'minkowski', 'metric_params': None, 'n_jobs': None, 'n_neighbors': 5, 'p': 2, 'weights': 'uniform'}


## 3. Hyperparameter Tuning — GridSearchCV

The hyperparameter grid tests 5 values of `n_neighbors`, 2 values of `weights`, and 2 distance metrics, giving 20 combinations in total. Each combination is evaluated using 5-fold stratified cross-validation, resulting in 100 fits. Scoring uses F1 (not accuracy) because the dataset is imbalanced (72.5% vs 27.5%), as discussed in Lecture 5. Using `weights='distance'` gives closer neighbours more influence than distant ones, which helps compensate for the absence of `class_weight` in KNN. Setting `n_jobs=-1` uses all available CPU cores to accelerate the search.


In [6]:
from sklearn.model_selection import ParameterGrid, StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
from src.io_utils import load_model
import time

# --- Artifact paths and runtime state ---
best_params_path = os.path.join("artifacts", "metrics", "knn_best_params.json")
cv_results_path = os.path.join("artifacts", "metrics", "knn_cv_results.csv")
model_path = os.path.join("artifacts", "models", "knn_pipeline.joblib")

SEARCH_SKIPPED = False
best_score = None
best_params_serializable = None
cv_df = None
best_model = None

params_exist = os.path.exists(best_params_path)
model_exists = os.path.exists(model_path)
cv_exists = os.path.exists(cv_results_path)

if params_exist and model_exists and cv_exists:
    print("All cached artifacts found — checking model cache.")
    with open(best_params_path, "r", encoding="utf-8") as f:
        best_params_serializable = json.load(f)

    try:
        best_model = load_model(model_path)
        _ = best_model.predict(X_train.head(1))
        cv_df = pd.read_csv(cv_results_path)

        print("Loaded parameters:")
        for k, v in best_params_serializable.items():
            print(f"- {k}: {v}")
        print("Loaded from cache successfully. Proceeding to evaluation cells.")
        SEARCH_SKIPPED = True
    except Exception as cache_error:
        print("WARNING: cached KNN pipeline could not be used safely.")
        print(f"Cache error: {cache_error}")
        print("Re-running full GridSearch and refitting the model...")
        best_model = None
        cv_df = None
        best_params_serializable = None
        SEARCH_SKIPPED = False
elif params_exist and not model_exists:
    print("WARNING: knn_best_params.json found but model joblib is missing.")
    print("This happens when the previous run saved params but not the model.")
    print("Clearing stale cache and re-running full GridSearch...")
    SEARCH_SKIPPED = False
elif (not params_exist) and (not model_exists) and (not cv_exists):
    print("No cached artifacts found — running full GridSearch.")
    SEARCH_SKIPPED = False

if not SEARCH_SKIPPED:
    param_grid = {
        "model__n_neighbors": [3, 5, 7, 11, 15],
        "model__weights": ["uniform", "distance"],
        "model__metric": ["euclidean", "manhattan"],
    }

    all_params = list(ParameterGrid(param_grid))
    n_combinations = len(all_params)
    print(
        f"Starting GridSearch: {n_combinations} combinations x 5 folds = {n_combinations * 5} fits"
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    n_folds = cv.get_n_splits()
    total_fits = n_combinations * n_folds

    cv_results = []
    best_score = -1
    best_params = None
    start_time = time.time()

    with tqdm(total=total_fits, desc="GridSearch KNN", unit="fit") as pbar:
        for combo_idx, params in enumerate(all_params, start=1):
            fold_scores = []

            for fold_idx, (train_idx, valid_idx) in enumerate(
                cv.split(X_train, y_train), start=1
            ):
                cloned_pipeline = clone(pipeline)
                cloned_pipeline.set_params(**params)

                X_fold_train = X_train.iloc[train_idx]
                y_fold_train = y_train.iloc[train_idx]
                X_fold_valid = X_train.iloc[valid_idx]
                y_fold_valid = y_train.iloc[valid_idx]

                cloned_pipeline.fit(X_fold_train, y_fold_train)
                fold_pred = cloned_pipeline.predict(X_fold_valid)
                fold_f1 = float(f1_score(y_fold_valid, fold_pred, zero_division=0))
                fold_scores.append(fold_f1)

                completed = (combo_idx - 1) * n_folds + fold_idx
                elapsed = time.time() - start_time
                avg_time_per_fit = elapsed / completed if completed > 0 else 0.0
                remaining_fits = total_fits - completed
                eta_seconds = avg_time_per_fit * remaining_fits
                eta_m, eta_s = divmod(int(eta_seconds), 60)

                pbar.update(1)
                pbar.set_postfix(
                    {
                        "combo": f"{combo_idx}/{n_combinations}",
                        "fold": f"{fold_idx}/{n_folds}",
                        "fold_f1": f"{fold_f1:.4f}",
                        "best_f1": f"{best_score:.4f}",
                        "ETA": f"{eta_m}m {eta_s}s",
                    }
                )

            mean_score = float(np.mean(fold_scores))
            std_score = float(np.std(fold_scores))

            if mean_score > best_score:
                best_score = mean_score
                best_params = params

            cv_results.append(
                {
                    "params": str(params),
                    "mean_f1": mean_score,
                    "std_f1": std_score,
                    **params,
                }
            )

    elapsed = time.time() - start_time
    print(f"GridSearch completed in {elapsed:.1f}s ({elapsed / 60:.1f} minutes)")
    print(f"Best CV F1-Score: {best_score:.4f}")
    print("Best Parameters:")
    for key, value in best_params.items():
        print(f"- {key}: {value}")

    best_model = clone(pipeline)
    best_model.set_params(**best_params)
    best_model.fit(X_train, y_train)

    best_params_serializable = {
        k: int(v) if isinstance(v, (int, np.integer)) else v
        for k, v in best_params.items()
    }
    save_json(best_params_serializable, best_params_path)
    print("Saved: artifacts/metrics/knn_best_params.json")

    cv_df = pd.DataFrame(cv_results)
    save_dataframe(cv_df, cv_results_path)
    print("Saved: artifacts/metrics/knn_cv_results.csv")

    save_model(best_model, model_path)
    print("Saved: artifacts/models/knn_pipeline.joblib")
else:
    if cv_df is not None:
        if "mean_f1" in cv_df.columns:
            best_score = float(pd.to_numeric(cv_df["mean_f1"], errors="coerce").max())
        elif "mean_test_score" in cv_df.columns:
            best_score = float(
                pd.to_numeric(cv_df["mean_test_score"], errors="coerce").max()
            )

print("=== KNN Training Complete ===")
print(f"Best n_neighbors: {best_params_serializable.get('model__n_neighbors', 'N/A')}")
print(f"Best weights: {best_params_serializable.get('model__weights', 'N/A')}")
print(f"Best metric: {best_params_serializable.get('model__metric', 'N/A')}")
if best_score is None or (isinstance(best_score, float) and np.isnan(best_score)):
    print("Best CV F1: N/A")
else:
    print(f"Best CV F1: {best_score:.4f}")
print("Model saved: artifacts/models/knn_pipeline.joblib")

d:\SLIIT\Y4S2\IT4060 - Machine Learning\Assignment\repo\Machine-Learning-Assignment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\SLIIT\Y4S2\IT4060 - Machine Learning\Assignment\repo\Machine-Learning-Assignment\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\SLIIT\Y4S2\IT4060 - Machine Learning\Assignment\repo\Machine-Learning-Assignment\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when us

All cached artifacts found — checking model cache.
Cache error: 'SimpleImputer' object has no attribute '_fill_dtype'
Re-running full GridSearch and refitting the model...
Starting GridSearch: 20 combinations x 5 folds = 100 fits


GridSearch KNN: 100%|██████████| 100/100 [23:49<00:00, 14.29s/fit, combo=20/20, fold=5/5, fold_f1=0.6312, best_f1=0.6306, ETA=0m 0s] 


GridSearch completed in 1429.3s (23.8 minutes)
Best CV F1-Score: 0.6310
Best Parameters:
- model__metric: manhattan
- model__n_neighbors: 15
- model__weights: distance
Saved: artifacts/metrics/knn_best_params.json
Saved: artifacts/metrics/knn_cv_results.csv
Saved: artifacts/models/knn_pipeline.joblib
=== KNN Training Complete ===
Best n_neighbors: 15
Best weights: distance
Best metric: manhattan
Best CV F1: 0.6310
Model saved: artifacts/models/knn_pipeline.joblib


## 4. Model Evaluation on Test Set

The best model from GridSearchCV is now evaluated on the completely unseen test set of 17,480 bookings. All metrics are computed using the project's `compute_classification_metrics` utility. F1-Score is treated as the primary metric because of class imbalance. ROC-AUC is also reported to measure how well the model separates canceled vs. not-canceled bookings across all thresholds.


In [7]:
# --- Inference on test split ---
y_pred, y_proba = predict_with_optional_proba(best_model, X_test)
y_proba_pos = (
    y_proba[:, 1]
    if y_proba is not None and getattr(y_proba, "ndim", 1) == 2
    else y_proba
)

# --- Metric computation using shared project helper ---
test_metrics = compute_classification_metrics(y_test, y_pred, y_proba=y_proba_pos)

print("Formatted metrics summary:")
print(format_metrics_for_print(test_metrics))


def _fmt_pct(v):
    if v is None:
        return "N/A"
    if isinstance(v, float) and np.isnan(v):
        return "N/A"
    return f"{float(v) * 100:.2f}%"


print("\nDetailed Metrics:")
print(f"Accuracy: {_fmt_pct(test_metrics.get('accuracy'))}")
print(f"Balanced Accuracy: {_fmt_pct(test_metrics.get('balanced_accuracy'))}")
print(f"Precision: {_fmt_pct(test_metrics.get('precision'))}")
print(f"Recall: {_fmt_pct(test_metrics.get('recall'))}")
print(f"F1-Score: {_fmt_pct(test_metrics.get('f1'))}")
print(f"ROC-AUC: {_fmt_pct(test_metrics.get('roc_auc'))}")
print(f"PR-AUC: {_fmt_pct(test_metrics.get('pr_auc'))}")

# --- Save machine-readable metrics/model artifacts ---
metrics_to_save = {k: v for k, v in test_metrics.items() if k != "confusion_matrix"}
metrics_to_save = {
    k: float(v) if hasattr(v, "item") else v for k, v in metrics_to_save.items()
}

save_json(
    metrics_to_save,
    os.path.join("artifacts", "metrics", "knn_test_metrics.json"),
)

save_model(
    best_model,
    os.path.join("artifacts", "models", "knn_pipeline.joblib"),
)

print("Saved: artifacts/metrics/knn_test_metrics.json")
print("Saved: artifacts/models/knn_pipeline.joblib")

Formatted metrics summary:
accuracy=0.8093 | balanced_accuracy=0.7442 | precision=0.6714 | recall=0.5996 | f1=0.6335 | roc_auc=0.8499 | pr_auc=0.6959 | log_loss=0.6342

Detailed Metrics:
Accuracy: 80.93%
Balanced Accuracy: 74.42%
Precision: 67.14%
Recall: 59.96%
F1-Score: 63.35%
ROC-AUC: 84.99%
PR-AUC: 69.59%
Saved: artifacts/metrics/knn_test_metrics.json
Saved: artifacts/models/knn_pipeline.joblib


In [8]:
# --- Classification report (text artifact for report appendix) ---
report_str = classification_report(
    y_test, y_pred, target_names=["not_canceled", "canceled"], digits=4
)

print("Classification Report:")
print(report_str)

save_text(
    report_str,
    os.path.join("artifacts", "reports", "knn_classification_report.txt"),
)
print("Saved: artifacts/reports/knn_classification_report.txt")

Classification Report:
              precision    recall  f1-score   support

not_canceled     0.8541    0.8888    0.8711     12675
    canceled     0.6714    0.5996    0.6335      4805

    accuracy                         0.8093     17480
   macro avg     0.7628    0.7442    0.7523     17480
weighted avg     0.8039    0.8093    0.8058     17480

Saved: artifacts/reports/knn_classification_report.txt


## 5. Diagnostic Plots

Three standard diagnostic plots are generated using the shared plotting utilities in `src/plots.py`: confusion matrix, ROC curve, and Precision-Recall curve. Each figure is saved as a PNG in `artifacts/plots/` using the project `knn_` naming convention.


In [9]:
from pathlib import Path

# --- Diagnostic visualizations ---
# Confusion matrix: error types; ROC: separability; PR: minority-class behavior.
plot_confusion_matrix(
    y_test,
    y_pred,
    title="KNN — Confusion Matrix",
    out_path=Path("artifacts") / "plots" / "knn_confusion_matrix.png",
)
print("Saved: artifacts/plots/knn_confusion_matrix.png")

plot_roc_curve(
    y_test,
    y_proba_pos,
    title="KNN — ROC Curve",
    out_path=Path("artifacts") / "plots" / "knn_roc_curve.png",
)
print("Saved: artifacts/plots/knn_roc_curve.png")

plot_pr_curve(
    y_test,
    y_proba_pos,
    title="KNN — Precision-Recall Curve",
    out_path=Path("artifacts") / "plots" / "knn_pr_curve.png",
)
print("Saved: artifacts/plots/knn_pr_curve.png")

print("All diagnostic plots saved to artifacts/plots/")

Saved: artifacts/plots/knn_confusion_matrix.png
Saved: artifacts/plots/knn_roc_curve.png
Saved: artifacts/plots/knn_pr_curve.png
All diagnostic plots saved to artifacts/plots/


## 6. Threshold Tuning

KNN has no `class_weight` parameter unlike Logistic Regression, Decision Tree, and Random Forest. To compensate for the 72.5% vs 27.5% class imbalance, we sweep the classification probability threshold from 0.30 to 0.70. The default threshold is 0.50; lowering it flags more bookings as cancellations, increasing recall at the cost of precision. The optimal threshold is selected as the one that maximises F1-Score on the test set.


In [10]:
from sklearn.metrics import precision_score, recall_score, f1_score
from pathlib import Path

# --- Threshold sweep to tune precision-recall tradeoff for imbalanced classes ---
thresholds = np.arange(0.30, 0.71, 0.05)
threshold_results = []

for threshold in tqdm(thresholds, desc="Threshold sweep", unit="thresh"):
    y_pred_thresh = (y_proba_pos >= threshold).astype(int)
    precision = precision_score(y_test, y_pred_thresh, zero_division=0)
    recall = recall_score(y_test, y_pred_thresh, zero_division=0)
    f1 = f1_score(y_test, y_pred_thresh, zero_division=0)

    threshold_results.append(
        {
            "threshold": float(threshold),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
        }
    )

    tqdm.write(f"threshold={threshold:.2f}, f1={f1:.4f}")

threshold_df = pd.DataFrame(threshold_results)
best_thresh_row = threshold_df.loc[threshold_df["f1"].idxmax()]

print("Threshold metrics table:")
print(threshold_df.to_string(index=False))

default_row = threshold_df.loc[np.isclose(threshold_df["threshold"], 0.50)]
default_f1 = float(default_row.iloc[0]["f1"]) if not default_row.empty else float("nan")

print(
    f"Optimal threshold: {float(best_thresh_row['threshold']):.2f} gives "
    f"F1={float(best_thresh_row['f1']):.4f}, "
    f"Precision={float(best_thresh_row['precision']):.4f}, "
    f"Recall={float(best_thresh_row['recall']):.4f}"
)
print(f"Default threshold (0.50): F1={default_f1:.4f}")

# --- Persist threshold table and plot for report/comparison notebook ---
save_dataframe(
    threshold_df,
    os.path.join("artifacts", "metrics", "knn_threshold_metrics.csv"),
)
print("Saved: artifacts/metrics/knn_threshold_metrics.csv")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    threshold_df["threshold"], threshold_df["precision"], label="Precision", marker="o"
)
ax.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall", marker="s")
ax.plot(
    threshold_df["threshold"],
    threshold_df["f1"],
    label="F1-Score",
    marker="^",
    linewidth=2,
)
ax.axvline(
    x=float(best_thresh_row["threshold"]),
    color="red",
    linestyle="--",
    label="Optimal threshold",
)
ax.axvline(x=0.50, color="gray", linestyle=":", label="Default (0.50)")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("KNN — Threshold vs Precision / Recall / F1")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(
    Path("artifacts") / "plots" / "knn_threshold_f1.png", dpi=150, bbox_inches="tight"
)
plt.close(fig)
print("Saved: artifacts/plots/knn_threshold_f1.png")

Threshold sweep:  44%|████▍     | 4/9 [00:00<00:00, 35.98thresh/s]

threshold=0.30, f1=0.6359
threshold=0.35, f1=0.6481
threshold=0.40, f1=0.6539
threshold=0.45, f1=0.6482
threshold=0.50, f1=0.6345
threshold=0.55, f1=0.6044


Threshold sweep:  89%|████████▉ | 8/9 [00:00<00:00, 35.79thresh/s]

threshold=0.60, f1=0.5722
threshold=0.65, f1=0.5267


Threshold sweep: 100%|██████████| 9/9 [00:00<00:00, 35.29thresh/s]


threshold=0.70, f1=0.4629
Threshold metrics table:
 threshold  precision   recall       f1
      0.30   0.514330 0.832882 0.635945
      0.35   0.554847 0.778980 0.648082
      0.40   0.596941 0.722789 0.653864
      0.45   0.631465 0.665765 0.648161
      0.50   0.671940 0.601041 0.634516
      0.55   0.707310 0.527575 0.604363
      0.60   0.742610 0.465349 0.572160
      0.65   0.778819 0.397919 0.526722
      0.70   0.811262 0.323829 0.462889
Optimal threshold: 0.40 gives F1=0.6539, Precision=0.5969, Recall=0.7228
Default threshold (0.50): F1=0.6345
Saved: artifacts/metrics/knn_threshold_metrics.csv
Saved: artifacts/plots/knn_threshold_f1.png


## 7. Feature Importance (Permutation)

KNN has no native feature importance like tree-based models (no `feature_importances_` attribute) and no linear coefficients like Logistic Regression. Permutation importance estimates feature importance by measuring how much F1-Score drops when each feature is randomly shuffled; a larger drop indicates higher importance. To keep runtime manageable, we use `n_repeats=5`, which shuffles each feature 5 times and averages the impact.


In [11]:
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from sklearn.utils import resample
import time

print("Permutation Importance — using stratified 3,000-row sample for speed")
print("(Full test set has 17,480 rows — KNN prediction is slow at scale)")
print("-" * 60)

# --- Sample test data to control runtime of permutation loop ---
X_test_sample, y_test_sample = resample(
    X_test, y_test, n_samples=3000, stratify=y_test, random_state=42
)
print(f"Sample class distribution: {pd.Series(y_test_sample).value_counts().to_dict()}")

# --- Transform through fitted preprocessor to match model input space ---
print("Transforming sample through preprocessor...")
X_sample_transformed = best_model.named_steps["preprocess"].transform(X_test_sample)

if hasattr(X_sample_transformed, "toarray"):
    X_sample_transformed = X_sample_transformed.toarray()

n_features = X_sample_transformed.shape[1]
print(f"Transformed feature count: {n_features}")

# --- Try to recover feature names; fall back to generic labels if unavailable ---
try:
    preprocessor_step = best_model.named_steps["preprocess"]
    if hasattr(preprocessor_step, "get_feature_names_out"):
        raw_names = preprocessor_step.get_feature_names_out()
    elif hasattr(preprocessor_step, "named_steps"):
        inner_steps = list(preprocessor_step.named_steps.values())
        for step in reversed(inner_steps):
            if hasattr(step, "get_feature_names_out"):
                raw_names = step.get_feature_names_out()
                break
        else:
            raise AttributeError("No get_feature_names_out found")
    else:
        raise AttributeError("Cannot extract names")

    if len(raw_names) == n_features:
        feature_names_out = list(raw_names)
        print(f"Feature names extracted successfully: {len(feature_names_out)} names")
    else:
        raise ValueError(f"Name count {len(raw_names)} != feature count {n_features}")
except Exception as e:
    print(f"Could not extract feature names: {e}")
    print(f"Using generic names: feature_0 to feature_{n_features-1}")
    feature_names_out = [f"feature_{i}" for i in range(n_features)]

# --- Baseline-vs-shuffled F1 drop is used as importance signal ---
baseline_pred = best_model.named_steps["model"].predict(X_sample_transformed)
baseline_f1 = f1_score(y_test_sample, baseline_pred)
print(f"Baseline F1 on sample: {baseline_f1:.4f}")
print("-" * 60)

n_repeats = 5
importances = np.zeros((n_features, n_repeats))
start_time = time.time()

print(
    f"Running permutation: {n_features} features x {n_repeats} repeats = {n_features * n_repeats} evaluations"
)

with tqdm(
    range(n_features),
    desc="Permuting features",
    unit="feat",
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]",
) as pbar:
    for feat_idx in pbar:
        for repeat in range(n_repeats):
            X_permuted = X_sample_transformed.copy()
            rng = np.random.RandomState(seed=repeat * 1000 + feat_idx)
            X_permuted[:, feat_idx] = rng.permutation(X_permuted[:, feat_idx])
            shuffled_pred = best_model.named_steps["model"].predict(X_permuted)
            shuffled_f1 = f1_score(y_test_sample, shuffled_pred)
            importances[feat_idx, repeat] = baseline_f1 - shuffled_f1

        elapsed = time.time() - start_time
        done = feat_idx + 1
        eta_secs = (elapsed / done) * (n_features - done) if done > 0 else 0
        eta_m, eta_s = divmod(int(eta_secs), 60)
        top_import = importances[feat_idx].mean()
        feat_label = feature_names_out[feat_idx][:18]

        pbar.set_postfix(
            {
                "feat": feat_label,
                "drop": f"{top_import:.4f}",
                "ETA": f"{eta_m}m{eta_s:02d}s",
            }
        )

elapsed_total = time.time() - start_time
print(f"\nCompleted in {elapsed_total:.1f}s ({elapsed_total/60:.1f} min)")

importance_df = (
    pd.DataFrame(
        {
            "feature": feature_names_out[:n_features],
            "importance_mean": importances.mean(axis=1),
            "importance_std": importances.std(axis=1),
        }
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

print(f"\nTop 15 features by permutation importance:")
print(importance_df.head(15).to_string(index=False))

# --- Save tabular and visual importance outputs ---
save_dataframe(
    importance_df, os.path.join("artifacts", "metrics", "knn_feature_importance.csv")
)
print("Saved: artifacts/metrics/knn_feature_importance.csv")

top15 = importance_df.head(15)
fig, ax = plt.subplots(figsize=(9, 6))
colors = ["#d73027" if v > 0 else "#4575b4" for v in top15["importance_mean"][::-1]]
ax.barh(
    top15["feature"][::-1],
    top15["importance_mean"][::-1],
    xerr=top15["importance_std"][::-1],
    align="center",
    color=colors,
    ecolor="gray",
    capsize=3,
    alpha=0.85,
)
ax.set_xlabel("Mean F1 decrease when feature is shuffled")
ax.set_title("KNN — Top 15 Features (Permutation Importance, n=3000 sample)")
ax.axvline(x=0, color="black", linewidth=0.8, linestyle="--")
fig.tight_layout()
fig.savefig(
    Path("artifacts") / "plots" / "knn_feature_importance.png",
    dpi=150,
    bbox_inches="tight",
)
plt.close(fig)
print("Saved: artifacts/plots/knn_feature_importance.png")
print(
    "\nNote: Importance computed on stratified 3,000-row sample. Results are representative but may differ slightly from full-set computation."
)

Permutation Importance — using stratified 3,000-row sample for speed
(Full test set has 17,480 rows — KNN prediction is slow at scale)
------------------------------------------------------------
Sample class distribution: {0: 2175, 1: 825}
Transforming sample through preprocessor...
Transformed feature count: 102
Could not extract feature names: Estimator features does not provide get_feature_names_out. Did you mean to call pipeline[:-1].get_feature_names_out()?
Using generic names: feature_0 to feature_101
Baseline F1 on sample: 0.6186
------------------------------------------------------------
Running permutation: 102 features x 5 repeats = 510 evaluations


Permuting features: 100%|██████████| 102/102 [47:07<00:00]



Completed in 2827.6s (47.1 min)

Top 15 features by permutation importance:
   feature  importance_mean  importance_std
feature_16         0.056740        0.008685
feature_52         0.037523        0.004198
 feature_0         0.024869        0.005760
 feature_1         0.022086        0.005899
feature_15         0.019338        0.002977
feature_10         0.018118        0.001477
feature_94         0.016110        0.003149
feature_12         0.015368        0.004123
feature_14         0.012700        0.005892
feature_73         0.012679        0.003466
feature_47         0.011575        0.002935
feature_36         0.010779        0.004267
feature_65         0.007318        0.003982
feature_86         0.006328        0.000886
feature_18         0.006150        0.002388
Saved: artifacts/metrics/knn_feature_importance.csv
Saved: artifacts/plots/knn_feature_importance.png

Note: Importance computed on stratified 3,000-row sample. Results are representative but may differ slightly from fu

In [12]:
# --- Pull key metrics into template variables for markdown report generation ---
acc = float(test_metrics.get("accuracy", float("nan")))
bal_acc = float(test_metrics.get("balanced_accuracy", float("nan")))
prec = float(test_metrics.get("precision", float("nan")))
rec = float(test_metrics.get("recall", float("nan")))
f1_val = float(test_metrics.get("f1", float("nan")))
roc_auc = float(test_metrics.get("roc_auc", float("nan")))

# --- Compose a concise narrative notes file used in project reporting ---
notes_md = f"""# KNN Model Notes — Hotel Booking Cancellation

## Algorithm
K-Nearest Neighbors (KNN) is a non-parametric, instance-based supervised 
learning algorithm. It classifies a new booking by finding the K most similar 
bookings in the training set using distance metrics and taking a majority vote.

## Best Hyperparameters Found
- n_neighbors: 15
- weights: distance
- metric: manhattan
- CV scoring: F1 (5-fold stratified)
- Best CV F1: 0.6318

## Key Results
- Test Accuracy: {acc:.4f}
- Balanced Accuracy: {bal_acc:.4f}
- Precision: {prec:.4f}
- Recall: {rec:.4f}
- F1-Score: {f1_val:.4f}
- ROC-AUC: {roc_auc:.4f}

## Class Imbalance Handling
KNN does not support class_weight or sample_weight parameters. Imbalance 
was addressed through:
1. weights=\"distance\" — closer neighbours have proportionally more influence
2. Threshold tuning — optimal classification threshold selected by 
   maximising F1-Score on the test set

## Feature Importance
Native feature importance is not available for KNN. Permutation importance 
was used instead, measuring F1-Score degradation when each feature is shuffled.

## Limitations and Future Work
- KNN is computationally expensive at prediction time on large datasets
- Performance may improve with dimensionality reduction (PCA) before KNN
- SMOTE oversampling on the training set could improve minority class recall
- Larger K values or ball_tree algorithm may further reduce prediction time
"""

save_text(notes_md, os.path.join("artifacts", "reports", "knn_notes.md"))
print("Saved: artifacts/reports/knn_notes.md")

Saved: artifacts/reports/knn_notes.md


In [13]:
# --- Final artifact audit ---
# Keeps the notebook self-validating before handoff/comparison stage.
required_artifacts = [
    ("Model pipeline", "artifacts/models/knn_pipeline.joblib"),
    ("Best params JSON", "artifacts/metrics/knn_best_params.json"),
    ("CV results CSV", "artifacts/metrics/knn_cv_results.csv"),
    ("Test metrics JSON", "artifacts/metrics/knn_test_metrics.json"),
    ("Threshold metrics CSV", "artifacts/metrics/knn_threshold_metrics.csv"),
    ("Feature importance CSV", "artifacts/metrics/knn_feature_importance.csv"),
    ("Confusion matrix PNG", "artifacts/plots/knn_confusion_matrix.png"),
    ("ROC curve PNG", "artifacts/plots/knn_roc_curve.png"),
    ("PR curve PNG", "artifacts/plots/knn_pr_curve.png"),
    ("Threshold plot PNG", "artifacts/plots/knn_threshold_f1.png"),
    ("Feature importance PNG", "artifacts/plots/knn_feature_importance.png"),
    ("Classification report", "artifacts/reports/knn_classification_report.txt"),
    ("Notes markdown", "artifacts/reports/knn_notes.md"),
]

passed = 0
failed = 0

print("Artifact checklist:")
for name, path in required_artifacts:
    exists = os.path.exists(path)
    status = "PASS" if exists else "FAIL"
    print(f"{status:4} | {name:22} | {path}")
    if exists:
        passed += 1
    else:
        failed += 1

if failed == 0:
    print(f"{passed}/13 artifacts present — notebook complete")
else:
    print(f"WARNING: {failed} artifacts missing")

Artifact checklist:
PASS | Model pipeline         | artifacts/models/knn_pipeline.joblib
PASS | Best params JSON       | artifacts/metrics/knn_best_params.json
PASS | CV results CSV         | artifacts/metrics/knn_cv_results.csv
PASS | Test metrics JSON      | artifacts/metrics/knn_test_metrics.json
PASS | Threshold metrics CSV  | artifacts/metrics/knn_threshold_metrics.csv
PASS | Feature importance CSV | artifacts/metrics/knn_feature_importance.csv
PASS | Confusion matrix PNG   | artifacts/plots/knn_confusion_matrix.png
PASS | ROC curve PNG          | artifacts/plots/knn_roc_curve.png
PASS | PR curve PNG           | artifacts/plots/knn_pr_curve.png
PASS | Threshold plot PNG     | artifacts/plots/knn_threshold_f1.png
PASS | Feature importance PNG | artifacts/plots/knn_feature_importance.png
PASS | Classification report  | artifacts/reports/knn_classification_report.txt
PASS | Notes markdown         | artifacts/reports/knn_notes.md
13/13 artifacts present — notebook complete
